# Workbook 4: Bayesian optimisation with a prebuilt oracle

This notebook turns a fitted oracle into a **Bayesian optimisation problem**. Rather than rebuilding models, we start from a prebuilt oracle for `dataset_b4` and focus on the BO choices a chemist actually has to make: what to optimise, which conditions are allowed, how to encode categorical inputs, how to start the campaign, and how to balance exploration against exploitation.

The aim is to understand what each choice in BO means chemically and how it changes the optimisation campaign.


## 0. Learning objectives

By the end of this notebook you should be able to:

- identify the controllable inputs and measurable outputs in an oracle-based optimisation problem
- define a BO objective and a realistic search space
- explain the role of encoding, initial design, surrogate model, and acquisition rule
- interpret single-run and multi-run BO plots
- carry the same workflow into workbook 5 for a different dataset

**How to use this notebook:** run the cells top-to-bottom once with the defaults, then change one decision at a time and see what happens. Variables that can be changed are typically fully-capitalized.


## 1. Load the worked example

Load the prebuilt oracle and inspect its feature metadata. In chemistry terms, this answers two questions:

- What can we change? -> BO inputs / decision variables
- What do we care about? -> oracle outputs / targets

Here the worked example is Jensen case IV, with one categorical choice and three continuous variables.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from itertools import product
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    ExpSineSquared,
    Matern,
    RBF,
    RationalQuadratic,
    WhiteKernel,
)

from scripts import get_oracle_meta, load_oracle
from scripts.bo_optimisation import (
    best_so_far,
    build_search_space,
    run_bo,
    run_multiple_bo,
    stack_best_so_far,
    summarize_runs,
)

ORACLE_KEY = "dataset_b4"
ORACLE_META = get_oracle_meta(ORACLE_KEY)
bundle = load_oracle(ORACLE_KEY)
spec = bundle.spec

feature_summary = []
for feature in spec.features:
    feature_summary.append(
        {
            "feature": feature,
            "type": "categorical" if feature in spec.categorical else "continuous",
            "default_space": spec.categorical_levels[feature] if feature in spec.categorical else spec.bounds[feature],
        }
    )

print(f"Loaded oracle: {ORACLE_META['label']}")
print(f"Available targets: {spec.targets}")
pd.DataFrame(feature_summary)


## 2. Choose the objective and logging behavior

When treated simply, a BO campaign needs a single scalar objective for each run. This oracle has two outputs, `yield` and `turnover`, so you must pick one target to optimise at a time.

The log settings are optional, but useful when you want to compare campaigns, save a full experiment trace, or inspect which rows came from the initial design versus BO itself.


In [ ]:
OPTIMIZE_TARGET = "yield"  # Options for dataset_b4: "yield", "turnover".
# Note: the current BO setup treats OPTIMIZE_TARGET as a maximise objective.
if OPTIMIZE_TARGET not in spec.targets:
    raise ValueError(f"Choose OPTIMIZE_TARGET from {spec.targets}")

SAVE_EXPERIMENT_LOGS = False  # Options: True, False.
EXPERIMENT_LOG_DIR = "bo_experiment_logs"
SINGLE_RUN_LOG_NAME = f"{ORACLE_KEY}_single_run.csv"
MULTI_RUN_LOG_TEMPLATE = f"{ORACLE_KEY}_run_{{run_index:02d}}_experiment_log.csv"

pd.DataFrame(
    [
        {"setting": "OPTIMIZE_TARGET", "value": OPTIMIZE_TARGET},
        {"setting": "SAVE_EXPERIMENT_LOGS", "value": SAVE_EXPERIMENT_LOGS},
        {"setting": "EXPERIMENT_LOG_DIR", "value": EXPERIMENT_LOG_DIR},
        {"setting": "SINGLE_RUN_LOG_NAME", "value": SINGLE_RUN_LOG_NAME},
        {"setting": "MULTI_RUN_LOG_TEMPLATE", "value": MULTI_RUN_LOG_TEMPLATE},
    ]
)


## 3. Define the search space

The search space is the set of conditions BO is allowed to try. Leaving it at the default asks: *what is the best point anywhere in the dataset domain?* Narrowing it asks a different question: *what is the best point within my practical constraints?*

In a real campaign, this can be used to constrain the search space to e.g. exclude certain solvents or reduce the difficulty of the search problem.

In [ ]:
SEARCH_SPACE_OVERRIDE = {
    # "ligand": ["L0", "L2", "L4"],
    # "temperature": (50.0, 105.0),
}

SEARCH_SPACE = build_search_space(spec, SEARCH_SPACE_OVERRIDE)

pd.DataFrame(
    [
        {
            "feature": feature,
            "type": "categorical" if feature in spec.categorical else "continuous",
            "search_space": SEARCH_SPACE[feature],
        }
        for feature in spec.features
    ]
)


## 4. Choose the categorical encoding

Gaussian Processes need numerical values to optimise, so categorical chemistry choices must be numerically encoded before modelling. This matters because the encoding defines what the model thinks "similar" categories look like.

- `onehot`: safest default; categories are treated as distinct with no imposed ordering
- `ordinal`: simple but likely misleading
- `custom`: lets you insert real descriptors if you have them

One-hot encoding is very common and is identical to presenting the GP with a binary string corresponding to the order of the choices, e.g. 0001 and 0010 might represent choices 1 and 2, respectively. 

Ordinal uses numbers like 1,2,3 etc. to indicate different choices. Whilst it appears to be a suitable "low-information" and simple encoding, it inherently groups choices together by position. This can unintentionally bias the GP to only search for choices that are numerically-close but physically very different.

In [ ]:
ENCODING_MODE = "onehot"  # Options: "onehot", "ordinal", "custom"
CUSTOM_CATEGORICAL_MAPS = {
    # Example structure if you have real numeric descriptors:
    # "ligand": {
    #     "L0": [0.10, 1.20, 3.40],
    #     "L1": [0.25, 1.05, 3.10],
    #     "L2": [0.40, 0.95, 2.85],
    #     "L3": [0.55, 0.80, 2.60],
    #     "L4": [0.70, 0.65, 2.30],
    #     "L5": [0.85, 0.55, 2.10],
    #     "L6": [1.00, 0.45, 1.95],
    # }
}

def onehot_encode_level(level, ordered_levels):
    if level not in ordered_levels:
        raise ValueError(f"Unknown categorical level '{level}'")
    vector = np.zeros(len(ordered_levels), dtype=float)
    vector[ordered_levels.index(level)] = 1.0
    return vector

def ordinal_encode_level(level, ordered_levels):
    if level not in ordered_levels:
        raise ValueError(f"Unknown categorical level '{level}'")
    return float(ordered_levels.index(level))

def custom_encode_level(level, feature_map):
    if level not in feature_map:
        raise ValueError(f"Custom map is missing level '{level}'")
    vector = np.asarray(feature_map[level], dtype=float).reshape(-1)
    if vector.size == 0:
        raise ValueError(f"Custom map for '{level}' cannot be empty")
    return vector

class NotebookConditionEncoder:
    def __init__(self, spec, search_space, *, mode, custom_maps=None):
        self.feature_order = list(spec.features)
        self.continuous = [feature for feature in self.feature_order if feature in spec.continuous]
        self.categorical = [feature for feature in self.feature_order if feature in spec.categorical]
        self.search_space = search_space
        self.mode = mode
        self.custom_maps = custom_maps or {}

        if self.mode not in {"onehot", "ordinal", "custom"}:
            raise ValueError("ENCODING_MODE must be one of: onehot, ordinal, custom")
        if self.mode == "custom":
            for feature in self.categorical:
                if feature not in self.custom_maps:
                    raise ValueError(f"Missing custom descriptor map for '{feature}'")
                missing = [level for level in self.search_space[feature] if level not in self.custom_maps[feature]]
                if missing:
                    raise ValueError(f"Custom descriptor map for '{feature}' is missing levels: {missing}")

    def _encode_feature(self, feature, level):
        ordered_levels = list(self.search_space[feature])
        if self.mode == "onehot":
            return onehot_encode_level(level, ordered_levels)
        if self.mode == "ordinal":
            return np.array([ordinal_encode_level(level, ordered_levels)], dtype=float)
        return custom_encode_level(level, self.custom_maps[feature])

    def transform_one(self, conditions):
        numeric = np.array([float(conditions[feature]) for feature in self.continuous], dtype=float)
        categorical_vectors = [self._encode_feature(feature, conditions[feature]) for feature in self.categorical]
        if categorical_vectors:
            return np.concatenate([numeric, *categorical_vectors])
        return numeric

    def transform_many(self, conditions_list):
        if not conditions_list:
            return np.empty((0, 0), dtype=float)
        return np.vstack([self.transform_one(conditions) for conditions in conditions_list])

def normalize_custom_maps(custom_maps):
    normalized = {}
    for feature, feature_map in custom_maps.items():
        normalized[feature] = {}
        for level, values in feature_map.items():
            vector = [float(value) for value in values]
            if not vector:
                raise ValueError(f"Custom map for '{feature}:{level}' cannot be empty")
            normalized[feature][level] = vector
    return normalized

def onehot_encoding(spec, search_space):
    return NotebookConditionEncoder(spec, search_space, mode="onehot")

def ordinal_encoding(spec, search_space):
    return NotebookConditionEncoder(spec, search_space, mode="ordinal")

def custom_encoding(spec, search_space, custom_maps):
    return NotebookConditionEncoder(spec, search_space, mode="custom", custom_maps=custom_maps)

def choose_encoding(mode, custom_maps):
    if mode == "onehot":
        return {
            "encoder_factory": onehot_encoding,
            "description": "Treat each categorical level as independent.",
        }
    if mode == "ordinal":
        return {
            "encoder_factory": ordinal_encoding,
            "description": "Give each categorical level a simple integer code.",
        }
    if mode == "custom":
        normalized_maps = normalize_custom_maps(custom_maps)
        return {
            "encoder_factory": lambda spec, search_space: custom_encoding(spec, search_space, normalized_maps),
            "description": "Replace each categorical level with your own numeric descriptor vector.",
        }
    raise ValueError("ENCODING_MODE must be one of: onehot, ordinal, custom")

encoding_settings = choose_encoding(ENCODING_MODE, CUSTOM_CATEGORICAL_MAPS)

pd.DataFrame([
    {"setting": "ENCODING_MODE", "value": ENCODING_MODE},
    {"setting": "description", "value": encoding_settings["description"]},
    {"setting": "custom_maps_provided", "value": bool(CUSTOM_CATEGORICAL_MAPS)},
])


## 5. Choose the initial design

This is the first batch of experiments BO sees before it starts using a GP and an acquisition function.

A small initial set leaves more budget for BO, but may give a weak first model. A larger initial set is safer, but spends more of the campaign before BO starts. Different strategies spread those first experiments differently across the search space.


In [ ]:
INIT_STRATEGY = "random"  # Options: "random", "latin_hypercube", "space_filling", "grid"
N_INITIAL = 12

def condition_key(conditions, feature_order):
    return tuple(conditions[feature] for feature in feature_order)

def all_categorical(search_space):
    return all(isinstance(value, list) for value in search_space.values())

def random_condition(search_space, feature_order, rng):
    condition = {}
    for feature in feature_order:
        value = search_space[feature]
        if isinstance(value, list):
            sampled = rng.choice(value)
            condition[feature] = sampled.item() if hasattr(sampled, "item") else sampled
        else:
            lower, upper = value
            condition[feature] = float(lower) if lower == upper else float(rng.uniform(lower, upper))
    return condition

def sample_random_conditions(search_space, feature_order, n, rng, seen_keys=None):
    seen_keys = set() if seen_keys is None else seen_keys

    if all_categorical(search_space):
        all_keys = list(product(*[search_space[feature] for feature in feature_order]))
        remaining = [key for key in all_keys if key not in seen_keys]
        if len(remaining) > n:
            chosen = rng.choice(len(remaining), size=n, replace=False)
            remaining = [remaining[index] for index in np.atleast_1d(chosen)]
        for key in remaining:
            seen_keys.add(key)
        return [dict(zip(feature_order, key)) for key in remaining]

    conditions = []
    attempts = 0
    max_attempts = max(1000, n * 200)
    while len(conditions) < n and attempts < max_attempts:
        candidate = random_condition(search_space, feature_order, rng)
        key = condition_key(candidate, feature_order)
        attempts += 1
        if key in seen_keys:
            continue
        seen_keys.add(key)
        conditions.append(candidate)
    return conditions

def latin_hypercube_points(n, bounds, rng):
    samples = np.empty((n, len(bounds)), dtype=float)
    for index, (lower, upper) in enumerate(bounds):
        if lower == upper:
            samples[:, index] = float(lower)
            continue
        perm = rng.permutation(n)
        u = (perm + rng.random(n)) / n
        samples[:, index] = lower + u * (upper - lower)
    return samples

def grid_points(n, bounds, rng):
    if not bounds:
        return np.empty((n, 0), dtype=float)
    grid_size = int(np.ceil(n ** (1.0 / max(len(bounds), 1))))
    axes = []
    for lower, upper in bounds:
        axes.append(np.array([float(lower)]) if lower == upper else np.linspace(lower, upper, grid_size))
    mesh = np.meshgrid(*axes, indexing="ij")
    grid = np.stack(mesh, axis=-1).reshape(-1, len(bounds))
    if len(grid) > n:
        chosen = rng.choice(len(grid), size=n, replace=False)
        grid = grid[np.atleast_1d(chosen)]
    return grid

def maximin_points(pool, n, rng):
    if len(pool) <= n:
        return pool
    chosen = [int(rng.integers(len(pool)))]
    while len(chosen) < n:
        chosen_points = pool[chosen]
        distances = np.linalg.norm(pool[:, None, :] - chosen_points[None, :, :], axis=2)
        min_distance = np.min(distances, axis=1)
        min_distance[chosen] = -np.inf
        chosen.append(int(np.argmax(min_distance)))
    return pool[chosen]

def assemble_conditions(feature_order, continuous_features, categorical_features, continuous_samples, search_space, rng):
    conditions = []
    for row in continuous_samples:
        condition = {}
        continuous_map = {
            feature: float(row[index])
            for index, feature in enumerate(continuous_features)
        }
        for feature in feature_order:
            if feature in categorical_features:
                sampled = rng.choice(search_space[feature])
                condition[feature] = sampled.item() if hasattr(sampled, "item") else sampled
            else:
                condition[feature] = continuous_map[feature]
        conditions.append(condition)
    return conditions

def latin_hypercube_initial_design(spec, search_space, feature_order, n, rng, seen_keys=None):
    seen_keys = set() if seen_keys is None else seen_keys
    continuous_features = [feature for feature in feature_order if feature in spec.continuous]
    categorical_features = [feature for feature in feature_order if feature in spec.categorical]
    bounds = [search_space[feature] for feature in continuous_features]
    if not bounds or all_categorical(search_space):
        return sample_random_conditions(search_space, feature_order, n, rng, seen_keys)
    points = latin_hypercube_points(n, bounds, rng)
    proposed = assemble_conditions(feature_order, continuous_features, categorical_features, points, search_space, rng)
    accepted = []
    for candidate in proposed:
        key = condition_key(candidate, feature_order)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        accepted.append(candidate)
    if len(accepted) < n:
        accepted.extend(sample_random_conditions(search_space, feature_order, n - len(accepted), rng, seen_keys))
    return accepted[:n]

def space_filling_initial_design(spec, search_space, feature_order, n, rng, seen_keys=None):
    seen_keys = set() if seen_keys is None else seen_keys
    continuous_features = [feature for feature in feature_order if feature in spec.continuous]
    categorical_features = [feature for feature in feature_order if feature in spec.categorical]
    bounds = [search_space[feature] for feature in continuous_features]
    if not bounds or all_categorical(search_space):
        return sample_random_conditions(search_space, feature_order, n, rng, seen_keys)
    pool_size = max(n * 10, 200)
    pool = latin_hypercube_points(pool_size, bounds, rng)
    points = maximin_points(pool, n, rng)
    proposed = assemble_conditions(feature_order, continuous_features, categorical_features, points, search_space, rng)
    accepted = []
    for candidate in proposed:
        key = condition_key(candidate, feature_order)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        accepted.append(candidate)
    if len(accepted) < n:
        accepted.extend(sample_random_conditions(search_space, feature_order, n - len(accepted), rng, seen_keys))
    return accepted[:n]

def grid_initial_design(spec, search_space, feature_order, n, rng, seen_keys=None):
    seen_keys = set() if seen_keys is None else seen_keys
    continuous_features = [feature for feature in feature_order if feature in spec.continuous]
    categorical_features = [feature for feature in feature_order if feature in spec.categorical]
    bounds = [search_space[feature] for feature in continuous_features]
    if not bounds or all_categorical(search_space):
        return sample_random_conditions(search_space, feature_order, n, rng, seen_keys)
    points = grid_points(n, bounds, rng)
    proposed = assemble_conditions(feature_order, continuous_features, categorical_features, points, search_space, rng)
    accepted = []
    for candidate in proposed:
        key = condition_key(candidate, feature_order)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        accepted.append(candidate)
    if len(accepted) < n:
        accepted.extend(sample_random_conditions(search_space, feature_order, n - len(accepted), rng, seen_keys))
    return accepted[:n]

def random_initial_design(spec, search_space, feature_order, n, rng, seen_keys=None):
    return sample_random_conditions(search_space, feature_order, n, rng, seen_keys)

def choose_initial_design(strategy, n_initial):
    handlers = {
        "random": random_initial_design,
        "latin_hypercube": latin_hypercube_initial_design,
        "space_filling": space_filling_initial_design,
        "grid": grid_initial_design,
    }
    descriptions = {
        "random": "Sample initial experiments uniformly from the search space.",
        "latin_hypercube": "Spread continuous samples more evenly across each dimension.",
        "space_filling": "Bias the starting design toward points that are far apart.",
        "grid": "Use a coarse lattice over the continuous bounds.",
    }
    if strategy not in handlers:
        raise ValueError("Unknown INIT_STRATEGY")
    if n_initial < 1:
        raise ValueError("N_INITIAL must be at least 1")
    return {
        "initial_design_sampler": handlers[strategy],
        "description": descriptions[strategy],
        "n_initial": int(n_initial),
    }

initial_design_settings = choose_initial_design(INIT_STRATEGY, N_INITIAL)

pd.DataFrame([
    {"setting": "INIT_STRATEGY", "value": INIT_STRATEGY},
    {"setting": "N_INITIAL", "value": initial_design_settings["n_initial"]},
    {"setting": "description", "value": initial_design_settings["description"]},
])


## 6. Choose the GP surrogate

The GP surrogate is the model BO uses to estimate both mean performance and uncertainty. The kernel controls what kinds of response surfaces the GP prefers.

As a rough guide:

- `matern52`: robust default for many chemistry problems
- `matern32`: allows rougher functions
- `rbf`: assumes very smooth behavior
- `rq`: allows a mixture of length scales
- `periodic`: mainly useful when true periodic structure is expected

The key point is to see that kernel choice changes the surrogate's assumptions. There is significant research in building your own kernels which goes beyond the scope of these workbooks.


In [ ]:
KERNEL_NAME = "matern52"  # Options: "matern52", "matern32", "rbf", "rq", "periodic"

def matern52_kernel():
    return (
        ConstantKernel(1.0, (1e-3, 1e3)) * Matern(nu=2.5)
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    )

def matern32_kernel():
    return (
        ConstantKernel(1.0, (1e-3, 1e3)) * Matern(nu=1.5)
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    )

def rbf_kernel():
    return (
        ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0)
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    )

def rq_kernel():
    return (
        ConstantKernel(1.0, (1e-3, 1e3)) * RationalQuadratic(length_scale=1.0, alpha=1.0)
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    )

def periodic_kernel():
    return (
        ConstantKernel(1.0, (1e-3, 1e3)) * ExpSineSquared(length_scale=1.0, periodicity=1.0)
        + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    )

def choose_kernel(kernel_name):
    kernel_builders = {
        "matern52": matern52_kernel,
        "matern32": matern32_kernel,
        "rbf": rbf_kernel,
        "rq": rq_kernel,
        "periodic": periodic_kernel,
    }
    descriptions = {
        "matern52": "Smooth but not overly rigid; a good default for most BO examples.",
        "matern32": "Rougher than Matern 5/2, so it can react faster to local changes.",
        "rbf": "Very smooth prior assumptions across the search space.",
        "rq": "A scale-mixture kernel that can adapt to multiple length scales.",
        "periodic": "Useful only when you expect repeating structure in the response surface.",
    }
    if kernel_name not in kernel_builders:
        raise ValueError("Unknown KERNEL_NAME")

    def build_gp(random_state):
        return GaussianProcessRegressor(
            kernel=kernel_builders[kernel_name](),
            normalize_y=True,
            n_restarts_optimizer=1,
            random_state=random_state,
        )

    return {
        "kernel_name": kernel_name,
        "description": descriptions[kernel_name],
        "gp_builder": build_gp,
    }

kernel_settings = choose_kernel(KERNEL_NAME)

pd.DataFrame([
    {"setting": "KERNEL_NAME", "value": kernel_settings["kernel_name"]},
    {"setting": "description", "value": kernel_settings["description"]},
])


## 7. Choose the acquisition rule

The acquisition function decides **where BO samples next**. It converts the surrogate's predicted mean and uncertainty into a score for each candidate.

This is the central BO trade-off:

- exploitation: go where the model already predicts high performance
- exploration: go where the model is uncertain and might discover something better

`N_CANDIDATES` controls how many candidate points are scored at each step.

### Choose a batch size

`BATCH_SIZE` experiments are selected before receiving any of their results; 1 gives sequential BO. We use **Kriging believer**: temporarily use the GP's predicted response for each selected point to help choose the next. These temporary predictions are discarded when the batch results arrive. [Method reference](https://secondmind-labs.github.io/trieste/1.2.0/notebooks/batch_optimization.html).

`N_ITER` counts batches. Maximum budget: `N_INITIAL + N_ITER * BATCH_SIZE`. Compare 30 batches of 1 with 10 batches of 3 for the same budget. Keep `N_CANDIDATES >= BATCH_SIZE`; a finite space may finish early.


In [ ]:
ACQ_FUNCTION = "ei"  # Options: "ei", "pi", "ucb", "greedy", "thompson"
EI_XI = 0.01
UCB_BETA = 2.0
N_CANDIDATES = 2000
N_ITER = 30  # BO batches after the initial design.
BATCH_SIZE = 1  # Experiments selected before seeing any results from that batch.

if ACQ_FUNCTION not in {"ei", "pi", "ucb", "greedy", "thompson"}:
    raise ValueError("Unknown ACQ_FUNCTION")

def describe_acquisition_function(acq_function):
    descriptions = {
        "ei": "Expected improvement: balance mean reward with uncertainty.",
        "pi": "Probability of improvement: focus on beating the current best.",
        "ucb": "Upper confidence bound: explicit exploration via beta.",
        "greedy": "Exploit the GP mean only. Select where the mean is highest.",
        "thompson": "Sample from the GP posterior for a stochastic policy.",
    }
    if acq_function not in descriptions:
        raise ValueError("Unknown ACQ_FUNCTION")
    return descriptions[acq_function]

def expected_improvement(mean, std, best_y, xi):
    safe_std = np.where(std > 0, std, 1.0)
    improvement = mean - best_y - xi
    z = improvement / safe_std
    scores = improvement * norm.cdf(z) + safe_std * norm.pdf(z)
    return np.where(std > 0, scores, 0.0)

def probability_improvement(mean, std, best_y, xi):
    safe_std = np.where(std > 0, std, 1.0)
    improvement = mean - best_y - xi
    z = improvement / safe_std
    scores = norm.cdf(z)
    return np.where(std > 0, scores, 0.0)

def upper_confidence_bound(mean, std, beta):
    return mean + beta * std

def score_candidates(gp, X_candidates, y_train, rng=None):
    mean, std = gp.predict(X_candidates, return_std=True)
    best_y = float(np.max(y_train))
    if ACQ_FUNCTION == "ei":
        return expected_improvement(mean, std, best_y, EI_XI)
    if ACQ_FUNCTION == "pi":
        return probability_improvement(mean, std, best_y, EI_XI)
    if ACQ_FUNCTION == "ucb":
        return upper_confidence_bound(mean, std, UCB_BETA)
    if ACQ_FUNCTION == "greedy":
        return mean
    rng = np.random.default_rng() if rng is None else rng
    return rng.normal(mean, np.maximum(std, 1e-9))

pd.DataFrame([
    {"setting": "ACQ_FUNCTION", "value": ACQ_FUNCTION},
    {"setting": "description", "value": describe_acquisition_function(ACQ_FUNCTION)},
    {"setting": "EI_XI", "value": EI_XI},
    {"setting": "UCB_BETA", "value": UCB_BETA},
    {"setting": "N_CANDIDATES", "value": N_CANDIDATES},
    {"setting": "N_ITER", "value": N_ITER},
    {"setting": "BATCH_SIZE", "value": BATCH_SIZE},
    {"setting": "max_experiment_budget", "value": N_INITIAL + N_ITER * BATCH_SIZE},
])


## 8. Run one BO campaign

This cell runs a single optimisation campaign with the choices above. The first `N_INITIAL` rows in the log are the starting design, so their `source` is `initial_set` and their GP prediction fields are blank. Later rows have `source = bo`, meaning they were chosen by the acquisition rule.

The log records `batch` (0 for the initial design) and `batch_position`. Predictions use the model fitted before the batch. Oracle calls simulate a batch without requiring parallel hardware.


In [ ]:
SINGLE_RUN_SEED = 42

single_run = run_bo(
    bundle,
    target=OPTIMIZE_TARGET,
    search_space_override=SEARCH_SPACE_OVERRIDE,
    encoder_factory=encoding_settings["encoder_factory"],
    initial_design_sampler=initial_design_settings["initial_design_sampler"],
    gp_builder=kernel_settings["gp_builder"],
    score_candidates=score_candidates,
    n_initial=N_INITIAL,
    n_iter=N_ITER,
    batch_size=BATCH_SIZE,
    n_candidates=N_CANDIDATES,
    random_state=SINGLE_RUN_SEED,
    save_logs=SAVE_EXPERIMENT_LOGS,
    log_dir=EXPERIMENT_LOG_DIR,
    log_name=SINGLE_RUN_LOG_NAME,
)

print(f"Best {OPTIMIZE_TARGET}: {single_run.best_value:.3f}")
print("Best conditions:")
print(single_run.best_conditions)
display(single_run.log)


## 9. Inspect the single-run BO trajectory

Two curves are especially useful:

- **best-so-far**: the best objective value found up to each experiment
- **simple regret**: how far the current best is from the best value found anywhere in that run

A single run is useful for intuition, but it can also be misleading because BO inherently relies on some randomness.

With batches, experiment-index curves show results retrospectively in selection order. Results become available together at each batch end; intermediate points do not represent model updates.


In [ ]:
best_curve = best_so_far(single_run.log)
best_overall = float(np.max(best_curve))
simple_regret = best_overall - best_curve
x_axis = np.arange(1, len(best_curve) + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x_axis, best_curve, marker="o")
axes[0].set_xlabel("Experiment index")
axes[0].set_ylabel(f"Best {OPTIMIZE_TARGET}")
axes[0].set_title(f"BO convergence: {ORACLE_META['label']}")

axes[1].plot(x_axis, simple_regret, marker="o", color="C1")
axes[1].set_xlabel("Experiment index")
axes[1].set_ylabel("Simple regret")
axes[1].set_title("Single-run regret")

plt.tight_layout()
plt.show()



## 10. Repeat with multiple runs

Different seeds change the initial design and stochastic parts of the BO loop. Repeating the campaign shows whether a result is robust or just lucky.

In practice, multi-run summaries are usually more informative than a single trajectory.


In [ ]:
N_RUNS = 5
BASE_SEED = 42

multi_run = run_multiple_bo(
    bundle,
    target=OPTIMIZE_TARGET,
    search_space_override=SEARCH_SPACE_OVERRIDE,
    encoder_factory=encoding_settings["encoder_factory"],
    initial_design_sampler=initial_design_settings["initial_design_sampler"],
    gp_builder=kernel_settings["gp_builder"],
    score_candidates=score_candidates,
    n_initial=N_INITIAL,
    n_iter=N_ITER,
    batch_size=BATCH_SIZE,
    n_candidates=N_CANDIDATES,
    n_runs=N_RUNS,
    base_seed=BASE_SEED,
    save_logs=SAVE_EXPERIMENT_LOGS,
    log_dir=EXPERIMENT_LOG_DIR,
    log_name_template=MULTI_RUN_LOG_TEMPLATE,
)

run_summary = summarize_runs(multi_run)
run_summary


## 11. Inspect the multi-run behavior

The left plot summarizes best-so-far across runs using a mean line, a mean +/- 1 std band, and a min-max envelope. The right plot summarizes simple regret using the mean and mean +/- 1 std.

Read these plots together: a high mean best-so-far is good, but a very wide spread means the method is not yet reliable.


In [ ]:
best_curves = stack_best_so_far(multi_run)
mean_best = np.nanmean(best_curves, axis=0)
std_best = np.nanstd(best_curves, axis=0)
min_best = np.nanmin(best_curves, axis=0)
max_best = np.nanmax(best_curves, axis=0)

regret_curves = np.array([np.nanmax(curve) - curve for curve in best_curves], dtype=float)
mean_regret = np.nanmean(regret_curves, axis=0)
std_regret = np.nanstd(regret_curves, axis=0)

x_axis = np.arange(1, len(mean_best) + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x_axis, mean_best, color="C0", label="mean best-so-far")
axes[0].fill_between(
    x_axis,
    mean_best - std_best,
    mean_best + std_best,
    color="C0",
    alpha=0.18,
    label="mean +/- 1 std",
)
axes[0].fill_between(
    x_axis,
    min_best,
    max_best,
    color="0.6",
    alpha=0.12,
    label="min-max envelope",
)
axes[0].set_xlabel("Experiment index")
axes[0].set_ylabel(f"Best {OPTIMIZE_TARGET}")
axes[0].set_title(f"Multi-run BO summary: {ORACLE_META['label']}")
axes[0].legend()

axes[1].plot(x_axis, mean_regret, color="C1", label="mean regret")
axes[1].fill_between(
    x_axis,
    mean_regret - std_regret,
    mean_regret + std_regret,
    color="C1",
    alpha=0.18,
    label="mean +/- 1 std",
)
axes[1].set_xlabel("Experiment index")
axes[1].set_ylabel("Simple regret")
axes[1].set_title("Multi-run regret")
axes[1].legend()

plt.tight_layout()
plt.show()

run_summary


## 12. What to take forward

Once the worked example makes sense, workbook 5 uses the same BO machinery in a more open-ended way. The questions should stay the same:

- What am I optimising?
- What conditions are actually allowed?
- How are categories represented?
- How much budget do I spend on the initial design?
- Does the BO behavior look robust across repeated runs?
